# Bonus B5 solution, notebook 1: the manager says no, and a request that asks for too much

The worked answers to the two challenge tasks of `b5a_meet_agent_framework.ipynb`. The setup repeats the notebook's
steps and its agent, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


The calendar keyed by request id, the step that asks the manager and the step that books, as in Parts B and D.

In [2]:
from typing import Never
from agent_framework import Executor, WorkflowBuilder, WorkflowContext, executor, handler, response_handler

# The team's calendar, keyed by the id of the request
BOOKINGS = {}


class AskManager(Executor):
    @handler
    async def ask(self, request: dict, ctx: WorkflowContext[dict]):
        print("STEP RAN: ask_manager asks")
        await ctx.request_info(request_data=request, response_type=bool)

    @response_handler
    async def answer(self, request: dict, approved: bool, ctx: WorkflowContext[dict, str]):
        print("STEP RAN: ask_manager received the answer", approved)
        if approved:
            await ctx.send_message(request)
        else:
            await ctx.yield_output(f"Not approved. Nothing booked for {request['name']}.")


@executor(id="book_days")
async def book_days_once(request: dict, ctx: WorkflowContext[Never, str]):
    print("STEP RAN: book_days")
    if request["id"] in BOOKINGS:
        await ctx.yield_output(f"Request {request['id']} is already booked. Nothing added.")
    else:
        BOOKINGS[request["id"]] = request
        await ctx.yield_output(f"Booked {request['days']} days for {request['name']}.")

The leave records, the tool, the agent that writes the note and the step that runs it, as in Part E.

In [3]:
from openai import AsyncOpenAI
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient

client = OpenAIChatClient(model=MODEL, async_client=AsyncOpenAI(timeout=60, max_retries=2))
OPTIONS = {"reasoning": {"effort": EFFORT}}

LEAVE_RECORDS = {"Sara": 12, "Omar": 4}


def days_left(name: str) -> str:
    """Look up how many days of annual leave an employee has left this year."""
    print(f"TOOL CALLED: days_left({name})")
    if name not in LEAVE_RECORDS:
        return f"No match for an employee named {name}."
    return f"{name} has {LEAVE_RECORDS[name]} days of annual leave left this year."


note_writer = Agent(
    client=client,
    name="Note writer",
    instructions=(
        "You write short notes to a manager about a leave request. Two sentences. "
        "Use days_left and say how many days of leave the employee has left."
    ),
    tools=[days_left],
    default_options=OPTIONS,
)


@executor(id="draft_note")
async def draft_note(request: dict, ctx: WorkflowContext[dict]):
    print("STEP RAN: draft_note")
    answer = await note_writer.run(
        f"{request['name']} asks for {request['days']} days of annual leave from {request['start']}. "
        + request.get("comment", "")
    )
    await ctx.send_message({**request, "note": answer.text})

---

## Task 1: The manager says no

The workflow of Part E: `draft_note`, then `ask_manager`, then `book_days`. We send a new request, `leave-3`, and
read the note the manager is shown.

In [4]:
ask_manager = AskManager(id="ask_manager")

builder = WorkflowBuilder(name="leave_with_note", start_executor=draft_note)
builder.add_edge(draft_note, ask_manager)
builder.add_edge(ask_manager, book_days_once)
workflow = builder.build()

sara = {"id": "leave-3", "name": "Sara", "days": 5, "start": "4 January"}

run = await workflow.run(sara)
question = run.get_request_info_events()[0]

print()
print("The manager is shown:")
print(question.data["note"])

STEP RAN: draft_note


TOOL CALLED: days_left(Sara)


STEP RAN: ask_manager asks

The manager is shown:
Sara has 12 days of annual leave left this year. She requests 5 days off starting 4 January.


The workflow is waiting at `ask_manager`. This time the answer is `False`.

In [5]:
run = await workflow.run(responses={question.request_id: False})

print("outputs: ", run.get_outputs())
print("state:   ", run.get_final_state())
print("bookings:", BOOKINGS)

STEP RAN: ask_manager received the answer False
outputs:  ['Not approved. Nothing booked for Sara.']
state:    WorkflowRunState.IDLE
bookings: {}


`ask_manager` received `False` and ended the workflow with the output `Not approved. Nothing booked for Sara.` It
sent no message along the edge, so no `STEP RAN: book_days` line printed and `BOOKINGS` is still empty. The state is
`IDLE`: the workflow finished, with a no.

---

## Task 2: Stop a request that asks for too much, before anyone is asked

`check_days` is a step like the others. It reads the leave records in code. When the request asks for more days
than are left, it ends the workflow with an output; otherwise it sends the request on. `WorkflowContext[dict, str]`
says it may do either: send a dictionary to the next step, or give a string as the workflow's result.

In [6]:
@executor(id="check_days")
async def check_days(request: dict, ctx: WorkflowContext[dict, str]):
    print("STEP RAN: check_days")
    left = LEAVE_RECORDS[request["name"]]
    if request["days"] > left:
        await ctx.yield_output(f"Not enough days: {request['name']} asks for {request['days']} and has {left} left.")
    else:
        await ctx.send_message(request)


ask_manager = AskManager(id="ask_manager")

builder = WorkflowBuilder(name="leave_checked", start_executor=check_days)
builder.add_edge(check_days, ask_manager)
builder.add_edge(ask_manager, book_days_once)
workflow = builder.build()

Omar has 4 days left and asks for 6.

In [7]:
run = await workflow.run({"id": "leave-4", "name": "Omar", "days": 6, "start": "21 December"})

print("outputs:          ", run.get_outputs())
print("questions waiting:", len(run.get_request_info_events()))
print("state:            ", run.get_final_state())

STEP RAN: check_days
outputs:           ['Not enough days: Omar asks for 6 and has 4 left.']
questions waiting: 0
state:             WorkflowRunState.IDLE


Only `check_days` ran. One output says there are not enough days, no question is waiting and the state is `IDLE`.
The manager was never asked: comparing the days asked for with the days in `LEAVE_RECORDS` is a rule our code
can check, so it needs neither a person nor a model.

The same workflow with a request for 2 days:

In [8]:
run = await workflow.run({"id": "leave-5", "name": "Omar", "days": 2, "start": "21 December"})

print("questions waiting:", len(run.get_request_info_events()))
print("state:            ", run.get_final_state())
print("bookings:         ", BOOKINGS)

STEP RAN: check_days
STEP RAN: ask_manager asks
questions waiting: 1
state:             WorkflowRunState.IDLE_WITH_PENDING_REQUESTS
bookings:          {}


`check_days` sent the request on, `ask_manager` asked, and the workflow waits with one question. Nothing is booked
until a person answers.

Back to [the notebook](../b5a_meet_agent_framework.ipynb).